# Spam Model Comparison Demo

This notebook compares three architectures on the same Kaggle spam dataset:

- `embeddingbag`
- `lstm`
- `transformer`

It reuses the repo's preprocessing, vocabulary, `LightningDataModule`, and `LightningModule` code paths.

## Environment and learning goals

This is a local-first notebook. Use it after the baseline notebook if you want to compare runtime, parameter count, and validation/test metrics across the three architectures.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
candidates = [cwd / 'requirements.txt', cwd.parent / 'requirements.txt']
requirements_path = next((path for path in candidates if path.exists()), None)
if requirements_path is None:
    raise FileNotFoundError('Could not find requirements.txt from the current notebook working directory.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])
print(f'Installed requirements from {requirements_path}')

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / 'src').exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / 'src').exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError('Could not locate the repo root containing src/.')

for path in (REPO_ROOT / 'src', REPO_ROOT / 'scripts'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f'REPO_ROOT = {REPO_ROOT}')

In [ ]:
from spam_lightning.config import ProjectConfig
from spam_lightning.utils.paths import ensure_dir

defaults = ProjectConfig()
KAGGLE_DATASET_SLUG = defaults.preprocess.dataset_slug
RAW_DIR = ensure_dir(REPO_ROOT / defaults.paths.raw_dir)
PROCESSED_DIR = ensure_dir(REPO_ROOT / defaults.paths.processed_dir)
ARTIFACTS_DIR = ensure_dir(REPO_ROOT / defaults.paths.artifacts_dir)
LOGS_DIR = ensure_dir(REPO_ROOT / defaults.paths.logs_dir)
LABEL_MAP_OVERRIDE = {}
SEED = defaults.preprocess.seed

print(f'KAGGLE_DATASET_SLUG = {KAGGLE_DATASET_SLUG}')
print(f'PROCESSED_DIR = {PROCESSED_DIR}')

In [ ]:
from download_kaggle import download_dataset
from spam_lightning.data.preprocessing import preprocess_dataset

tabular_files = download_dataset(dataset=KAGGLE_DATASET_SLUG, raw_dir=RAW_DIR)
preprocess_result = preprocess_dataset(
    raw_dir=RAW_DIR,
    out_dir=PROCESSED_DIR,
    label_map=LABEL_MAP_OVERRIDE or None,
    seed=SEED,
    dataset_slug=KAGGLE_DATASET_SLUG,
)
print('Discovered tabular files:')
for path in tabular_files:
    print(f' - {path}')
preprocess_result

In [ ]:
import json

stats = json.loads((PROCESSED_DIR / 'stats.json').read_text(encoding='utf-8'))
print(json.dumps(stats, indent=2))

In [ ]:
from spam_lightning.data.datamodule import SpamDataModule

bag_datamodule = SpamDataModule(
    data_dir=PROCESSED_DIR,
    batch_size=defaults.data.batch_size,
    num_workers=0,
    pin_memory=False,
    lowercase=stats['lowercase'],
    min_freq=defaults.data.min_freq,
    max_vocab_size=defaults.data.max_vocab_size,
    model_name='embeddingbag',
    max_seq_len=defaults.data.max_seq_len,
)
bag_datamodule.setup('fit')
bag_batch = next(iter(bag_datamodule.train_dataloader()))
print('EmbeddingBag batch keys:', list(bag_batch.keys()))
print('tokens.shape =', tuple(bag_batch['tokens'].shape))
print('offsets.shape =', tuple(bag_batch['offsets'].shape))
print('labels.shape =', tuple(bag_batch['labels'].shape))

In [ ]:
sequence_datamodule = SpamDataModule(
    data_dir=PROCESSED_DIR,
    batch_size=defaults.data.batch_size,
    num_workers=0,
    pin_memory=False,
    lowercase=stats['lowercase'],
    min_freq=defaults.data.min_freq,
    max_vocab_size=defaults.data.max_vocab_size,
    model_name='lstm',
    max_seq_len=defaults.data.max_seq_len,
)
sequence_datamodule.setup('fit')
sequence_batch = next(iter(sequence_datamodule.train_dataloader()))
print('Sequence batch keys:', list(sequence_batch.keys()))
print('input_ids.shape =', tuple(sequence_batch['input_ids'].shape))
print('attention_mask.shape =', tuple(sequence_batch['attention_mask'].shape))
print('lengths.shape =', tuple(sequence_batch['lengths'].shape))
print('labels.shape =', tuple(sequence_batch['labels'].shape))

In [ ]:
MODEL_RUNS = [
    {'model_name': 'embeddingbag', 'max_epochs': 5},
    {'model_name': 'lstm', 'max_epochs': 5},
    {'model_name': 'transformer', 'max_epochs': 5},
]
MODEL_RUNS

In [ ]:
import json
import time

import pandas as pd
import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from spam_lightning.data.datamodule import SpamDataModule
from spam_lightning.models.lit_model import SpamLitModule
from spam_lightning.utils.paths import ensure_dir
from spam_lightning.utils.seed import set_global_seed


def to_float(value):
    if hasattr(value, 'item'):
        return float(value.item())
    return float(value)


results = []
for run_cfg in MODEL_RUNS:
    model_name = run_cfg['model_name']
    run_name = f'{model_name}-comparison'
    run_artifacts_dir = ensure_dir(ARTIFACTS_DIR / run_name)

    set_global_seed(SEED)
    datamodule = SpamDataModule(
        data_dir=PROCESSED_DIR,
        batch_size=defaults.data.batch_size,
        num_workers=0,
        pin_memory=False,
        lowercase=stats['lowercase'],
        min_freq=defaults.data.min_freq,
        max_vocab_size=defaults.data.max_vocab_size,
        model_name=model_name,
        max_seq_len=defaults.data.max_seq_len,
    )
    datamodule.setup('fit')
    vocab_path = run_artifacts_dir / 'vocab.json'
    datamodule.save_vocab(vocab_path)

    model = SpamLitModule(
        vocab_size=datamodule.vocab_size,
        model_name=model_name,
        embed_dim=defaults.model.embed_dim,
        learning_rate=defaults.model.learning_rate,
        pad_index=datamodule.pad_index,
        dropout=defaults.model.dropout,
        lstm_hidden_dim=defaults.model.lstm_hidden_dim,
        lstm_num_layers=defaults.model.lstm_num_layers,
        lstm_bidirectional=defaults.model.lstm_bidirectional,
        transformer_num_layers=defaults.model.transformer_num_layers,
        transformer_num_heads=defaults.model.transformer_num_heads,
        transformer_ff_dim=defaults.model.transformer_ff_dim,
        transformer_pooling=defaults.model.transformer_pooling,
        transformer_positional_encoding=defaults.model.transformer_positional_encoding,
    )

    checkpoint_callback = ModelCheckpoint(
        monitor='val_f1',
        mode='max',
        save_top_k=1,
        filename='best',
        dirpath=str(run_artifacts_dir),
    )
    trainer = L.Trainer(
        accelerator='auto',
        devices='auto',
        max_epochs=run_cfg['max_epochs'],
        deterministic=True,
        precision='32-true',
        callbacks=[
            checkpoint_callback,
            EarlyStopping(monitor='val_f1', mode='max', patience=3),
            LearningRateMonitor(logging_interval='epoch'),
        ],
        logger=CSVLogger(save_dir=str(LOGS_DIR), name='spam_lightning', version=run_name),
    )

    started = time.perf_counter()
    trainer.fit(model, datamodule=datamodule)
    train_seconds = time.perf_counter() - started
    val_metrics = dict(trainer.callback_metrics)
    test_results = trainer.test(model, datamodule=datamodule, ckpt_path='best')
    test_metrics = test_results[0] if test_results else {}
    parameter_count = sum(parameter.numel() for parameter in model.parameters())

    config_payload = {
        'dataset_slug': KAGGLE_DATASET_SLUG,
        'run_name': run_name,
        'model_name': model_name,
        'selected_raw_file': stats.get('source_file'),
        'text_col': stats.get('text_col'),
        'label_col': stats.get('label_col'),
        'label_mapping': stats.get('label_mapping', {}),
        'split_ratios': stats.get('split_ratios'),
        'seed': SEED,
        'lowercase': stats.get('lowercase', True),
        'vocab': {
            'min_freq': defaults.data.min_freq,
            'max_vocab_size': defaults.data.max_vocab_size,
            'vocab_path': str(vocab_path.resolve()),
        },
        'model': {
            'model_name': model_name,
            'embed_dim': defaults.model.embed_dim,
            'learning_rate': defaults.model.learning_rate,
            'dropout': defaults.model.dropout,
            'vocab_size': datamodule.vocab_size,
            'pad_index': datamodule.pad_index,
            'lstm_hidden_dim': defaults.model.lstm_hidden_dim,
            'lstm_num_layers': defaults.model.lstm_num_layers,
            'lstm_bidirectional': defaults.model.lstm_bidirectional,
            'transformer_num_layers': defaults.model.transformer_num_layers,
            'transformer_num_heads': defaults.model.transformer_num_heads,
            'transformer_ff_dim': defaults.model.transformer_ff_dim,
            'transformer_pooling': defaults.model.transformer_pooling,
            'transformer_positional_encoding': defaults.model.transformer_positional_encoding,
            'max_seq_len': defaults.data.max_seq_len,
        },
        'trainer': {
            'max_epochs': run_cfg['max_epochs'],
            'precision': '32-true',
            'deterministic': True,
            'batch_size': defaults.data.batch_size,
            'num_workers': 0,
            'pin_memory': False,
            'max_seq_len': defaults.data.max_seq_len,
        },
        'artifacts': {
            'artifact_dir': str(run_artifacts_dir.resolve()),
            'best_ckpt': checkpoint_callback.best_model_path,
            'vocab_json': str(vocab_path.resolve()),
            'config_json': str((run_artifacts_dir / 'config.json').resolve()),
        },
    }
    (run_artifacts_dir / 'config.json').write_text(json.dumps(config_payload, indent=2), encoding='utf-8')

    results.append(
        {
            'model_name': model_name,
            'params': parameter_count,
            'train_seconds': round(train_seconds, 2),
            'val_acc': to_float(val_metrics.get('val_acc', 0.0)),
            'val_f1': to_float(val_metrics.get('val_f1', 0.0)),
            'test_acc': to_float(test_metrics.get('test_acc', 0.0)),
            'test_f1': to_float(test_metrics.get('test_f1', 0.0)),
            'best_ckpt': checkpoint_callback.best_model_path,
        }
    )

comparison_df = pd.DataFrame(results).sort_values(by='test_f1', ascending=False).reset_index(drop=True)
comparison_df

## Interpretation

Use the result table to discuss trade-offs:

- `embeddingbag` is usually the fastest and simplest baseline.
- `lstm` adds word-order information with moderate complexity.
- `transformer` adds self-attention and usually the most implementation complexity in this repo.

On small classroom datasets, the fastest model may still be the most practical one.